In [ ]:
import re
import pandas as pd

# 1) Path to your Sitemap log file
logfile = r"D:\sitemap_PSMA\sitemap_PSMA.log"

# 2) Read the entire log
with open(logfile, "r") as f:
    txt = f.read()

# 3) Regex to capture each Mapping block, its center/ranges, and the following SiteScore
pattern = re.compile(
    r"-- Mapping site\s+(\d+).*?"                                      # site number
    r"ligand xcm,ycm,zcm,xrange,yrange,zrange\s*"
    r"([-\d\.]+)\s+([-\d\.]+)\s+([-\d\.]+)\s+([-\d\.]+)\s+([-\d\.]+)\s+([-\d\.]+)"  # centers + half-ranges
    r".*?SiteScore.*?\n\s*([-\d\.]+)",                                   # capture the first number on the SiteScore line
    re.DOTALL
)

matches = pattern.findall(txt)

# 4) Build DataFrame
df = pd.DataFrame(matches, columns=[
    "site_rank",
    "center_x","center_y","center_z",
    "xrange","yrange","zrange",
    "score"
]).astype({
    "site_rank": int,
    "center_x": float, "center_y": float, "center_z": float,
    "xrange": float,   "yrange": float,   "zrange": float,
    "score": float
})

# 5) Keep only the top-2 sites by rank
top2 = df[df["site_rank"].isin([1, 2])].sort_values("site_rank")

# 6) Compute full box sizes for Vina
top2["size_x"] = top2["xrange"] * 2
top2["size_y"] = top2["yrange"] * 2
top2["size_z"] = top2["zrange"] * 2

# 7) Annotate with filename and a simple site label
top2["filename"] = logfile
top2["site"]     = top2["site_rank"].apply(lambda x: f"site_{x}")

# 8) Reorder columns
out = top2[[
    "filename","site","site_rank","score",
    "center_x","center_y","center_z",
    "size_x","size_y","size_z"
]]

# 9) Write out
out_csv = r"D:\sitemap_PSMA\top2_sites.csv"
out.to_csv(out_csv, index=False)

print(f"Wrote top 2 sites (with scores) to {out_csv}\n")
print(out)


In [14]:
import re
import pandas as pd
from pathlib import Path

# 1) Top‐level batch folder:
base_dir = Path(r"D:\Cross_docking_sitemap\sitemap_batch")

# 2) Regexes (same as before)
map_pat = re.compile(
    r"--\s*Mapping\s+site\s+(\d+).*?"
    r"ligand\s+xcm,ycm,zcm,xrange,yrange,zrange\s*"
    r"([-\d\.]+)\s+([-\d\.]+)\s+([-\d\.]+)\s+([-\d\.]+)\s+([-\d\.]+)\s+([-\d\.]+)",
    re.IGNORECASE|re.DOTALL
)
score_pat = re.compile(
    r"SiteScore.*?\n\s*([0-9]+(?:\.[0-9]+)?)",
    re.IGNORECASE|re.DOTALL
)

rows = []

for prep in base_dir.glob("*_prep"):
    logfile = next(prep.glob("*.log"), None)
    if logfile is None:
        # no log → skip entirely
        continue

    txt    = logfile.read_text()
    maps   = map_pat.findall(txt)
    scores = score_pat.findall(txt)

    # if no mapping sites found, skip
    if not maps:
        continue

    # loop over _all_ sites (1..N)
    for idx, grp in enumerate(maps):
        # grp = (site_rank, cx, cy, cz, xr, yr, zr)
        _, cx, cy, cz, xr, yr, zr = grp

        # fetch the corresponding score if present
        try:
            score = float(scores[idx])
        except (IndexError, ValueError):
            score = None

        cx, cy, cz = map(float, (cx, cy, cz))
        xr, yr, zr = map(float, (xr, yr, zr))

        rows.append({
            "protein":   prep.name.replace("_prep",""),
            "score":     score,
            "center_x":  cx,
            "center_y":  cy,
            "center_z":  cz,
            "size_x":    xr * 2,
            "size_y":    yr * 2,
            "size_z":    zr * 2,
            "logfile":   str(logfile)
        })

# assemble and write
if not rows:
    print("⚠️  No sites found in any log.")
else:
    master = pd.DataFrame(rows)
    out_csv = r"all_sites.csv"
    master.to_csv(out_csv, index=False)
    print(f"✅ Wrote {len(master)} rows to {out_csv}")


✅ Wrote 7767 rows to all_sites.csv


In [16]:
import re
import pandas as pd
from pathlib import Path

# 1) Top-level batch folder:
base_dir = Path(r"D:\Cross_docking_sitemap\sitemap_batch")

# 2) Regexes
map_pat = re.compile(
    r"--\s*Mapping\s+site\s+(\d+).*?"
    r"ligand\s+xcm,ycm,zcm,xrange,yrange,zrange\s*"
    r"([-\d\.]+)\s+([-\d\.]+)\s+([-\d\.]+)\s+([-\d\.]+)\s+([-\d\.]+)\s+([-\d\.]+)",
    re.IGNORECASE | re.DOTALL
)

score_pat = re.compile(
    r"SiteScore.*?\n\s*([0-9]+(?:\.[0-9]+)?)",
    re.IGNORECASE | re.DOTALL
)

grid_pat = re.compile(
    r"grid point counts\s+(\d+)\s+(\d+)\s+(\d+)",
    re.IGNORECASE
)

rows = []

for prep in base_dir.glob("*_prep"):
    logfile = next(prep.glob("*.log"), None)
    if logfile is None:
        continue

    txt = logfile.read_text()

    maps   = map_pat.findall(txt)
    scores = [float(s) for s in score_pat.findall(txt)]
    grids  = grid_pat.findall(txt)

    if not maps:
        continue

    for idx, grp in enumerate(maps):
        # unpack the regex groups
        _, cx, cy, cz, xr, yr, zr = grp
        cx, cy, cz = map(float, (cx, cy, cz))
        xr, yr, zr = map(float, (xr, yr, zr))

        # half-range → full box size
        box_x = xr * 2
        box_y = yr * 2
        box_z = zr * 2

        # score if available
        score = scores[idx] if idx < len(scores) else None

        # actual grid-point counts
        if idx < len(grids):
            gx, gy, gz = map(int, grids[idx])
        else:
            gx = gy = gz = None

        rows.append({
            "protein":   prep.name.replace("_prep",""),
            "site_rank": int(grp[0]),
            "score":     score,
            "center_x":  cx,
            "center_y":  cy,
            "center_z":  cz,
            # renamed from size_→box_
            "box_x":     box_x,
            "box_y":     box_y,
            "box_z":     box_z,
            # renamed from grid_n→size_
            "size_x":    gx,
            "size_y":    gy,
            "size_z":    gz,
            "logfile":   str(logfile)
        })

if not rows:
    print("⚠️  No sites found in any log.")
else:
    master = pd.DataFrame(rows)
    out_csv = r"all_sites_2.csv"
    master.to_csv(out_csv, index=False)
    print(f"✅ Wrote {len(master)} rows to {out_csv}")


✅ Wrote 7767 rows to all_sites_2.csv


In [18]:
#!/usr/bin/env python3
"""
extract_sitemap_boxes.py

Scan every *_prep/ folder inside `base_dir` and write a CSV with:

- protein      – UniProt code (folder stem without '_prep')
- site_rank    – 1 … n (order reported by SiteMap)
- score        – SiteScore
- center_x/y/z – Å
- box_x/y/z    – Å  (2 × xrange|yrange|zrange)
- size_x/y/z   – integer grid‑point counts along each axis
- logfile      – absolute path to the .log processed
"""

import re
import pandas as pd
from pathlib import Path

# --------------------------------------------------------------------------- #
# 1) top‑level batch directory
base_dir = Path(r"D:\Cross_docking_sitemap\sitemap_batch")

# --------------------------------------------------------------------------- #
# 2) robust regex patterns
FLOAT    = r"[-+]?\d+(?:\.\d+)?"

#    -- Mapping site N --
#    ligand xcm,ycm,zcm,xrange,yrange,zrange  <six floats>
map_pat = re.compile(
    rf"--\s*Mapping\s+site\s+(\d+).*?"
    rf"ligand\s+xcm,ycm,zcm,xrange,yrange,zrange\s*"
    rf"({FLOAT})\s+({FLOAT})\s+({FLOAT})\s+({FLOAT})\s+({FLOAT})\s+({FLOAT})",
    re.IGNORECASE | re.DOTALL
)

#    SiteScore size ...
#       1.028    89   …
score_pat = re.compile(
    r"SiteScore.*?\n\s*(" + FLOAT + r")",
    re.IGNORECASE | re.DOTALL
)

#    grid point counts   27   27   33
grid_pat = re.compile(
    r"grid point counts\s+(\d+)\s+(\d+)\s+(\d+)",
    re.IGNORECASE
)

# --------------------------------------------------------------------------- #
rows = []

for prep_dir in base_dir.glob("*_prep"):
    log = next(prep_dir.glob("*.log"), None)
    if not log:
        continue

    text = log.read_text()

    maps   = map_pat.findall(text)
    scores = [float(x) for x in score_pat.findall(text)]
    grids  = grid_pat.findall(text)

    if not maps:
        # malformed run or empty log
        continue

    for i, grp in enumerate(maps):
        rank, cx, cy, cz, xr, yr, zr = grp
        cx, cy, cz = map(float, (cx, cy, cz))
        xr, yr, zr = map(float, (xr, yr, zr))
        box_x, box_y, box_z = xr * 2, yr * 2, zr * 2

        # grid sizes (may be absent if SiteMap aborted before mapping)
        if i < len(grids):
            gx, gy, gz = map(int, grids[i])
        else:
            gx = gy = gz = None

        rows.append({
            "protein":   prep_dir.stem.replace("_prep", ""),
            "site_rank": int(rank),
            "score":     scores[i] if i < len(scores) else None,
            "center_x":  cx,
            "center_y":  cy,
            "center_z":  cz,
            "box_x":     box_x,
            "box_y":     box_y,
            "box_z":     box_z,
            "size_x":    gx,
            "size_y":    gy,
            "size_z":    gz,
            "logfile":   str(log)
        })

# --------------------------------------------------------------------------- #
if not rows:
    print("⚠️  No SiteMap sites found under", base_dir)
else:
    df = pd.DataFrame(rows)
    # optional sorting – keeps same order inside each protein
    df.sort_values(["protein", "site_rank"], inplace=True)
    out_csv = r"all_sites.csv"
    df.to_csv(out_csv, index=False)
    print(f"✅  Parsed {len(df)} sites → {out_csv}")


✅  Parsed 7767 sites → all_sites.csv
